In [1]:
import torch 
torch.cuda.empty_cache()

In [2]:
from lxml import etree
import json
import cv2
import numpy as np
from pathlib import Path
import os
from typing import List, Tuple

from datetime import datetime
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.mask_rcnn import (MaskRCNNPredictor, MaskRCNN_ResNet50_FPN_Weights)
from torchvision.models.detection.rpn import AnchorGenerator
from tqdm import tqdm

In [3]:
class SurgicalDatasetParser:
    def __init__(self,base_path,phase_json_path):
        self.base_path = Path(base_path)
        print(f"변환된 절대 경로 :{self.base_path}")
        self.label_path = self.base_path/"라벨링데이터"/"클래스1.(수술도구 이미지)"/"01.원위부위절제술"
        self.image_path = self.base_path/"원천데이터"/"클래스1.(수술도구 이미지)"/"01.원위부위절제술"
        self.tool_categories = self._discover_tool_categories()
        self.class_to_idx = {cat['name']: cat['id'] -1 for cat in self.tool_categories}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        print(f"발견된 수술 도구 카테고리:{len(self.tool_categories)}개")
        for cat in self.tool_categories:
            print(f"  ID {cat['id']-1} (orig_id {cat['id']}): {cat['name']}")
            
        #수술 단계 정보 로딩
        self.image_stem_to_phase_map = self._load_phase_data(phase_json_path)
        if self.image_stem_to_phase_map:
            self.phases = sorted(set(self.image_stem_to_phase_map.values()))
            self.phase_to_idx = {name : i for i, name in enumerate(self.phases)}
            self.idx_to_phase = {i : name for i, name in enumerate(self.phases)}
            self.num_phases = len(self.phases)

            print(f"발견된 수술 단계:{self.num_phases}개 - {self.phases}")
        else :
            print("경고: 수술 단계 정보를 로드하지 못했습니다.")
            self.phases, self.phase_to_idx, self.idx_to_phase, self.num_phases = [], {}, {}, 0


    #def _convert_to_torchvision_format(self, annotation_data, image_file, phase_id):
        
    
    
    def _discover_tool_categories(self):
        #데이터 셋에서 수술 도구 카데고리 자동 발견
        categories = []
        if not self.label_path.is_dir():
            print(f"경고 : 라벨 경로 '{self.label_path}'가 존재하지 않거나 디렉토리가 아닙니다.")
            return []
        try:
            tool_folders = [f for f in self.label_path.glob('*') if f.is_dir()]
            categories = [
                {
                    "id": idx + 1,
                    "name": folder.name,
                    "supercategory": "surgical_tool"
                }
                for idx, folder in enumerate(sorted(tool_folders))
            ]
        except OSError as e:
            print(f"치명적 오류 : 라벨 경로 '{self.label_path}'탐색 중 오류 발생. 경로를 확인 해주세요. (오류: {e})")

        return categories
        
    def _load_phase_data(self, phase_json_path):
        #JSON 파일에서 수술단계 정보를 로드,
        #JSON 파일은 각 항목리 "filename" 과 "phhase" 를 포함하는 리스트여한다. 
        image_stem_to_phase = {}
        if not phase_json_path:
            print("경고 : 수술 단계 정보 디렉토리 경로가 제공되지 않았습니다. 단계정보를 로드할 수 없습니다.") 
            return image_stem_to_phase
        phase_dir = Path(phase_json_path)

        if not phase_dir.is_dir():
            p#rint(f"경고 : 수술 단꼐 정보 디렉토리 '{phase_json_path}'을 찾을 수 없습니다.") 
            return image_stem_to_phase

        for json_file_path in phase_dir.glob("*.json"):
            try:
                with open(json_file_path, 'r', encoding ='utf-8') as f:
                    phase_annotations_list = json.load(f)
                if not isinstance(phase_annotations_list, list):
                    print(f"경고: 파일 '{json_file_path.name}'의 내용이 리스트가 아닙니다.")
                    continue
                for entry in phase_annotations_list:
                    if isinstance(entry, dict) and "filename" in entry and "phase" in entry:
                        image_stem_to_phase[Path(entry["filename"]).stem] = entry["phase"]
            except json.JSONDecodeError:
                print(f"경고 : 단계 정보 JSON 파일({phase_json_path})파싱 오류.")      
            except Exception as e:
                print(f"단계 정보 JSON 파일 로딩 중 예기치 않은 오류({json_file.name}) : {e}")  

        #print(f"총{successful_files_count}개의 파일에서 {total_entries_processed} 개의 단계 항목을 처리했습니다.")
        
        if not image_stem_to_phase:
            print(f"경고 : 단계 정보를 성공적으로 로드하지 못했거나, 로드된 정보가 없습니다. ({phase_json_path}) 경로와 파일 내용을 확인 해주세요.")  
        else:
            print(f"{len(image_stem_to_phase)}개의 고유 이미지 스템에 대한 단계 정보를 로드했씁니다.")
            
        return image_stem_to_phase
        
    def parse_xml_annotation(self, xml_file):
        try:
            tree = etree.parse(str(xml_file))
            root = tree.getroot()
            size_elem = root.find('size')
            

            annotation_data = {
                'filename': root.findtext('filename'),
                'size': {'width' :int(size_elem.findtext('width','0')), 'height' :int(size_elem.findtext('height','0')),'depth' :(size_elem.findtext('depth','0')) },
                'objects' : []
            }

            #객체 정보 추출
            for obj_elem in root.findall('object'):
                obj_data = self._parse_object_element(obj_elem)
                if obj_data:
                    annotation_data['objects'].append(obj_data)
                    
            return annotation_data

        except Exception as e:
            print(f"일반오류  in {xml_file} : {e}")
            return None

    def _parse_object_element(self,obj_elem):
        points = []           
        points_elem = obj_elem.find('points')
        if points_elem is not None:
            for x_elem, y_elem in zip(points_elem.findall('x'), points_elem.findall('y')):
                try:
                    points.append([float(x_elem.text),float(y_elem.text)])
                except (ValueError, TypeError):
                    continue
        if points: return{'name' :obj_elem.findtext('name'), 'points':points}
        return None

    def find_matching_image(self, xml_file, tool_folder):
        
        image_tool_folder = self.image_path / tool_folder
        #print(f"[DEBUG] 이미지 폴더 경로 : {image_tool_folder}")

        if not image_tool_folder.is_dir():
            print(f"[경고] 이미지 폴더를 찾을 수 없음 : {image_tool_folder}")
            return None
            
        try:
            for ext in ['*.png', '*.jpg','*.jpeg','*.PNG', '*.JPG','*.JPEG']:
                for image_file in image_tool_folder.glob(ext):
                    if image_file.stem == xml_file:
                        return image_file
        except OSError as e:
            print(f"[경고] 디렉토리 접근 오류. 건너뜁니다. 경로: {image_tool_folder} (오류 : {e})")
        return None
    
    def prepare_dataset_for_torchvision(self):
        dataset_items = []
        try :
            tool_dirs = [d for d in self.label_path.glob('*') if d.is_dir()]
        except OSError as e:
            print(f"치명적 오류: 기본 라벨 경로 {iself.label_path}  를 읽는 중 오류 발생 (오류 : {e})")
            return []
            
        for tool_dir in tqdm(tool_dirs, desc = "데이터 셋 아이템 준비 중") :
            tool_folder_name = tool_dir.name
            xml_files = list(tool_dir.glob("*.xml"))
            for xml_file_path in xml_files:
                annotation_data = self.parse_xml_annotation(xml_file_path)
                if not annotation_data or not annotation_data['objects']:
                    continue

                xml_file_stem = xml_file_path.stem
                image_file = self.find_matching_image(xml_file_stem, tool_folder_name)
                if not image_file:
                    #print(f"이미지 파일을 찾을 수 없음 1 : {xml_file}")
                    continue
                # 수술 단계 정보 조회 (XML 파일의 stem 기준)
                phase_label_str = self.image_stem_to_phase_map.get(xml_file_stem)
                if phase_label_str is None or phase_label_str not in self.phase_to_idx :
                    #print(f"단계 정보를 찾을 수 없음 : {xml_file_stem}")
                    continue
                if phase_label_str not in self.phase_to_idx:
                    print(f"경고 : {xml_file_stem}의 단계 '{phase_label_str} 가 정의 된 단계 목록에 없습니다.'")
                    continue
                phase_id = self.phase_to_idx[phase_label_str]
                
                item = self._convert_to_torchvision_format(annotation_data,image_file,phase_id,xml_file_stem)
                if item:
                    dataset_items.append(item)

        print(f"총 {len(dataset_items)}개 아이템 준비 완료")
        return dataset_items
        ##
    def _convert_to_torchvision_format(self, annotation_data, image_file_path, phase_id, image_stem):
        boxes = []
        labels = []
        masks = []
        img_width = annotation_data['size']['width']
        img_height = annotation_data['size']['height']

        for obj in annotation_data['objects']:
            if not obj['points'] or len(obj['points']) < 3:
                continue
                
            tool_name = obj['name']
            if tool_name not in self.class_to_idx : 
                continue
            
            #labels.append(self.class_to_idx[tool_name])
            points = np.array(obj['points'], dtype = np.float32)
            x_coords = points[:,0]
            y_coords = points[:,1]
            
            x_min, x_max = np.min(x_coords),np.max(x_coords)
            y_min, y_max = np.min(y_coords),np.max(y_coords)

            if x_max<=x_min or y_max<=y_min:
                print(f"[경고] 파일'{image_stem}'에서 유효하지 않은 바운딩 박스를 건너 뜁니다. :[x_min = {x_min},y_min = {y_min},x_max = {x_max},y_max = {y_max} ]")
                continue
            labels.append(self.class_to_idx[tool_name] +1) 
            boxes.append([x_min,y_min,x_max,y_max])
            masks.append(self._create_mask_from_points(obj['points'], img_width, img_height))
        if not boxes:
            return None
        return {
            'image_path':str(image_file_path),
            'boxes': np.array(boxes, dtype=np.float32),
            'labels': np.array(labels, dtype=np.int64),
            'masks': np.array(masks, dtype=np.uint8),
            'image_id': torch.tensor([hash(image_stem) % (2**32)]),
            'phase_id': phase_id
        }



    def _create_mask_from_points(self, points, width, height):
        """
        cv2.fillPoly 대신 Pillow(PIL)를 사용하여 마스크를 생성합니다.
        이렇게 하면 opencv-python-headless 패키지에서도 코드가 동작합니다.
        """
        # 1. 'L' 모드(8비트 흑백)의 비어있는(0으로 채워진) PIL 이미지를 생성합니다.
        mask_img = Image.new('L', (width, height), 0)
        
        # 2. PIL의 polygon 함수는 좌표를 튜플의 리스트 형태로 받으므로 변환합니다.
        #    예: [[x1, y1], [x2, y2]] -> [(x1, y1), (x2, y2)]
        pil_points = [tuple(p) for p in points]
        
        # 3. 생성된 이미지에 다각형을 그립니다. fill=1로 설정하여 내부를 1로 채웁니다.
        ImageDraw.Draw(mask_img).polygon(pil_points, outline=1, fill=1)
        
        # 4. PIL 이미지를 다시 NumPy 배열로 변환하여 반환합니다.
        mask = np.array(mask_img, dtype=np.uint8)
        return mask
        
base_data_path = "../../dataset/dataset001/179.6개암 최소침습수술 AI학습데이터/01.데이터/1.Training"
phase_json_path = "output_phase_labeling5"
#parser = SurgicalDatasetParser(base_data_path, phase_json_path)
#coco_data = parser.convert_to_coco_format("output/surgical_tools_coco.json")
#dataset_items = parser.prepare_dataset_for_torchvision()

In [4]:
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor:
    def __call__(self, image, target):
        image = T.functional.to_tensor(image)
        return image, target
        
class RandomHorizontalFlip:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image, target):
        if torch.rand(1) <self.p:
            image = T.functional.hflip(image)

            img_width, _=image.size
            
            if "boxes" in target:
                boxes = target["boxes"]
                boxes[:, [0,2]] = img_width - boxes[:, [2, 0]]
                target["boxes"] = boxes

            if "masks" in target:
                target["masks"] = target["masks"].flip(-1)
                #target["masks"] = T.functional.hflip(target["masks"])
        return image, target

class ColorJitter(T.ColorJitter):
  #  def __init__(self, brightness=0, contrast=0, saturation=0, hue=0):
  #      self.transform = T.Colorjitter(brightness, contrast, saturation, hue)

    def __call__(self, image, target):
        image = super().__call__(image)
        return image, target

class Normalize:
    def __init__(self, mean, std):
        #self.transform = T.Normalize(mean, std)
        self.mean = mean
        self.std = std

    def __call__(self, image, target):

        image = T.functional.normalize(image,mean=self.mean,std=self.std)
        return image, target
            
class SurgicalToolDataset(Dataset):
    def __init__(self, dataset_items, transforms_func=None):
        self.dataset_items = dataset_items
        self.transforms_func = transforms_func

    def __len__(self):
        return len(self.dataset_items)

    def __getitem__(self, idx):
        if isinstance(self.dataset_items, torch.utils.data.Subset):
            item = self.dataset_items.dataset[self.dataset_items.indices[idx]]
        else:
            item = self.dataset_items[idx]
            
        image = Image.open(item['image_path']).convert('RGB')

        target={}
        target['boxes'] = torch.as_tensor(item['boxes'], dtype = torch.float32)
        target['labels'] = torch.as_tensor(item['labels'], dtype = torch.int64)
        target['masks'] = torch.as_tensor(item['masks'], dtype = torch.uint8)
        target['image_id'] = item['image_id']

        #area 와 iscrowd (Mask R-CNN 학습에 필요)
        num_objs = len(item['boxes'])
        if num_objs > 0 :
            target['area'] = (target['boxes'][:,3] - target['boxes'][:,1] ) * (target['boxes'][:,2] - target['boxes'][:,0])
        else:
            target['area'] = torch.empty((0,),dtype=torch.float32)
            
        #target['masks'] = torch.as_tensor(item['masks'], dtype = torch.uint8)
        #target['area'] = (target['boxes'][:,3] - target['boxes'][:,1] ) * (target['boxes'][:,2] - target['boxes'][:,0])
        target['iscrowd'] = torch.zeros((num_objs,), dtype=torch.int64)
        #수술 단계 레이블 
        target['phase_label'] = torch.tensor(item['phase_id'], dtype=torch.int64)

        if self.transforms_func:
            image, target = self.transforms_func(image, target)
            
        return image, target
            
                                                                             
def get_tarnsform(train):
    #데이터 변환 함수
    transforms_list = []
    
    #transforms_list.append(T.ToTensor())
    if train:
        #데이터 증강 기법 추기
        transforms_list.append(ColorJitter(brightness=0.2, contrast =0.2, saturation=0.2, hue=0.1))
        
        #transforms_list.append(T.RandomHorizontalFlip(0.5))
        
    transforms_list.append(ToTensor())
    
    if train:
        pass
    transforms_list.append(Normalize(mean=[0.485,0.465,0.406],std=[0.229,0.224,0.225]))
    
        #transforms_list.append(T.PILToTensor())
    #transforms_list.append(T.Normalize(mean=[0.485,0.465,0.406],std=[0.229,0.224,0.225]))
    if train:
        final_transforms = [
            ColorJitter(brightness=0.2, contrast =0.2, saturation=0.2, hue=0.1),
            RandomHorizontalFlip(0.5),
            ToTensor(),
            #Normalize(mean=[0.485,0.465,0.406],std=[0.229,0.224,0.225])
        ]
    else:
        final_transforms = [
            ToTensor(),
            #Normalize(mean=[0.485,0.465,0.406],std=[0.229,0.224,0.225])
        ]
    return Compose(final_transforms)   
      

In [5]:
import torchvision, torch
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.rpn import RPNHead

#TorchVision Mask R-CNN 모델 구성(수술단계 예측 헤드 추가)
class MaskRCNNWithPhase(MaskRCNN):
    def __init__(self, backbone, num_tool_classes, num_phases, **kwargs):
        super().__init__(backbone, numclasses = None, **kwargs)
	# 각 도구의 존재 여부를 나타내는 특징
        self.tool_presence_features_dim = num_tool_classes
	# 각 도구 클래스의 평균 위치를 나타내는 특징
        self.tool_location_features_dim = num_tool_classes*4

        out_channels_fpn = backbone.out_channels #예 256 for ResNet50+FPN
        combined_feature_dim = (
            out_channels_fpn +
            self.tool_presence_features_dim +
            self.tool_location_features_dim
        )
        #수술 단계 분류 헤드
        phase_classifier_features =256 # 중간 FC 레이어 특징 수 (조정 가능)

        self.phase_global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.phase_fc1 = nn.Linear(combined_feature_dim, phase_classifier_features)
                #드롭아웃 레이어 추가
        self.phase_dropout = nn.Dropout(p=0.5)
        self.phase_classifier_head = nn.Linear(phase_classifier_features, num_phases)

        #self.phase_classifier_head = nn.Linear(phase_classifier_features, num_phases)
        self.num_tool_classes = num_tool_classes


    def forward(self, images, targets=None):
        # 수정된 forward 메서드 : 도구 감지 + 수술 단계 예측
        # images : List[Tensor]
        # target : List[Dict[str,Tensor]]
        original_image_sizes: List[Tuple[int,int]] = []
        for img in images:
            val = img.shape[-2:]
            original_image_sizes.append((val[0], val[1]))
        #MaskRCNN 의 transform 적용(이미지 정규화, 리사이징 등)
        images, targets = self.transform(images, targets) # targets 는  여기서 도구 탐지용
    
        #백본(FPN 포함)을 통해 특징맵 추출
        # ----전역 이미지 특징 추출
        features = self.backbone(images.tensors)

        #도구 감지(기존 MaskRCNN 로직)
        proposals, proposal_losses = self.rpn(images, features, targets)
        detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets)

        phase_feat_map = features.get('pool', features.get('0'))
        pooled_global_features = self.phase_global_pool(phase_feat_map).flatten(start_dim=1)


        if self.training:
            tool_presence_features, tool_location_features = self._get_tool_features_from_list(targets, images.image_sizes, is_gt=True)
           
        else:
            tool_presence_features, tool_location_features = self._get_tool_features_from_list(detections, images.image_sizes, is_gt=False)
    
        combined_features = torch.cat(
                [pooled_global_features,tool_presence_features,tool_location_features],dim=1
            )
        x_phase  =F.relu(self.phase_fc1(combined_features))
            #드롭아웃적용
        if self.training:
            x_phase  = self.phase_dropout(x_phase)
        phase_logits = self.phase_classifier_head(x_phase)
        
        losses = {}
        losses.update(detector_losses)
        losses.update(proposal_losses)
        
        if targets is not None: # 타겟이 있는 경우 (학습 또는 평가)
            phase_labels_list = [t["phase_label"] for t in targets]
            phase_labels_tensor = torch.stack(phase_labels_list).to(phase_logits.device)
            losses['loss_phase'] = F.cross_entropy(phase_logits, phase_labels_tensor)
            
        if self.training:
            return losses #학습시에는 loss 딕셔너리만 반환

        detections = self. transform.postprocess(detections, images.image_sizes, original_image_sizes)
        for i, det_result_dict in enumerate(detections):
            det_result_dict['phase_logits'] = phase_logits[i]
            det_result_dict['predicted_phase_idx'] = torch.argmax(phase_logits[i])
            det_result_dict['phase_probs'] = F.softmax(phase_logits[i], dim=-1)
        if targets is not None:
            return losses, detections

        return detections

    def _get_tool_features_from_list(self, data_list, image_sizes, is_gt = False):
        batch_presence_features = []
        batch_location_features = []

        device = data_list[0].get('boxes',torch.tensor([])).device
        
        for i, data in enumerate(data_list):
            image_height, image_width = image_sizes[i]
            
            if len(data['labels']) == 0:
                presence_vector = torch.zeros(self.num_tool_classes, device = device)
                location_vector = torch.zeros(self.num_tool_classes*4, device = device)
                batch_presence_features.append(presence_vector)
                batch_location_features.append(location_vector)
                continue
            if is_gt:
                fg_labels_indices = data['labels']-1
                fg_boxes = data['boxes']
            else:
                foreground_mask = data['labels'] > 0
                if not torch.any(foreground_mask):
                    presence_vector = torch.zeros(self.num_tool_classes, device=device)
                    location_vector = torch.zeros(self.num_tool_classes*4, device=device)
                    batch_presence_features.append(presence_vector)
                    batch_location_features.append(location_vector)
                    continue

                fg_labels = data['labels'][foreground_mask]
                fg_boxes = data['boxes'][foreground_mask]

                fg_labels_indices = fg_labels - 1

            presence_vector = torch.zeros(self.num_tool_classes, device=device)
            unique_labels_indices = torch.unique(fg_labels_indices)

            valid_indices_mask = (unique_labels_indices >= 0) & (unique_labels_indices < self.num_tool_classes)
            valid_unique_labels_indices = unique_labels_indices[valid_indices_mask]

            if valid_unique_labels_indices.numel() > 0:
                presence_vector.scatter_(0,valid_unique_labels_indices.long(),1.0)
                
            location_vector = torch.zeros(self.num_tool_classes*4, device = device)
                        
            for tool_idx in range(self.num_tool_classes):
                instances_mask = (fg_labels_indices==tool_idx)
                if torch.any(instances_mask):
                    boxes = fg_boxes[instances_mask]
                    norm_boxes_xywh = torch.stack([
                        (boxes[:, 0] + boxes[:, 2]) / 2 / image_width,
                        (boxes[:, 1] + boxes[:, 3]) / 2 / image_height,
                        (boxes[:, 2] - boxes[:, 0]) / image_width,
                        (boxes[:, 3] - boxes[:, 1]) / image_height
                    ], dim=1)
                    avg_box = torch.mean(norm_boxes_xywh, dim=0)
                    location_vector[tool_idx*4 : (tool_idx+1)*4] = avg_box
                    
            batch_presence_features.append(presence_vector)
            batch_location_features.append(location_vector)

        return torch.stack(batch_presence_features), torch.stack(batch_location_features)

def get_custom_surgical_mask_rcnn_model(
    num_tool_classes: int,
    num_phases: int,
    anchor_sizes=None,
    anchor_aspect_ratios=None,
    pretrained_coco_path: str = None,  # ← [수정] 로컬 COCO 가중치 경로를 받을 인자 추가
    freeze_backbone: bool = False,
    **kwargs
):
    # 1) 온라인 가중치 다운로드 관련 인자 제거
    for k in ['weights', 'weights_backbone', 'pretrained', 'pretrained_backbone']:
        kwargs.pop(k, None)

    # 2) 가중치 없이 기본 모델 구조 생성
    model_base = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, **kwargs
    )

    # 3) [수정] 로컬 COCO 가중치가 제공된 경우, 부분적으로 로드
    if pretrained_coco_path:
        # 다운로드된 가중치 파일 로드
        state_dict = torch.load(pretrained_coco_path, map_location='cpu')

        # load_state_dict의 strict=False를 사용하여 키가 완벽히 일치하지 않아도 로드
        # (예: box_predictor와 mask_predictor의 클래스 수가 달라 이름은 같아도 크기가 다름)
        # 이를 통해 백본, RPN 등 공통된 부분의 가중치만 성공적으로 불러올 수 있습니다.
        missing_keys, unexpected_keys = model_base.load_state_dict(state_dict, strict=False)
        
        print(f"로컬 COCO 가중치 로드 완료: {pretrained_coco_path}")
        print(f"  - 누락된 키 (대체로 정상): {len(missing_keys)}개 (e.g., roi_heads.box_predictor, roi_heads.mask_predictor)")
        print(f"  - 예기치 않은 키 (없어야 정상): {len(unexpected_keys)}개")

    # 4) (선택) 커스텀 앵커 설정 (기존과 동일)
    if anchor_sizes is not None and anchor_aspect_ratios is not None:
        anchor_gen = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_aspect_ratios)
        model_base.rpn.anchor_generator = anchor_gen
        per_loc = anchor_gen.num_anchors_per_location()
        if len(set(per_loc)) != 1:
            raise ValueError(f"num_anchors_per_location 불일치: {per_loc}")
        model_base.rpn.head = RPNHead(256, int(per_loc))

    # 5) [수정] 박스/마스크 예측기 헤드를 사용자 클래스 수에 맞게 교체 (전이 학습의 핵심)
    # 배경 클래스(+1)를 포함해야 합니다.
    in_features_box = model_base.roi_heads.box_predictor.cls_score.in_features
    model_base.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_tool_classes + 1)

    in_features_mask = model_base.roi_heads.mask_predictor.conv5_mask.in_channels
    model_base.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, 256, num_tool_classes + 1
    )
    
    # 6) (선택) 백본 동결 옵션
    if freeze_backbone:
        for parameter in model_base.backbone.parameters():
            parameter.requires_grad = False
        print("백본(backbone) 가중치를 동결했습니다. 헤드만 학습됩니다.")


    # 7) [수정] Phase 헤드가 포함된 전체 모델로 결합 (기존 코드의 오타 수정 포함)
    final_model = MaskRCNNWithPhase(
        model_base.backbone,
        num_tool_classes=num_tool_classes,
        num_phases=num_phases,
        rpn_anchor_generator=model_base.rpn.anchor_generator,
        box_roi_pool=model_base.roi_heads.box_roi_pool,  # ← [버그 수정] bfinal_transforms = ox_roi_pool 오타 수정
        box_head=model_base.roi_heads.box_head,
        box_predictor=model_base.roi_heads.box_predictor,
        mask_roi_pool=model_base.roi_heads.mask_roi_pool,
        mask_head=model_base.roi_heads.mask_head,
        mask_predictor=model_base.roi_heads.mask_predictor,
        rpn_head=model_base.rpn.head,
        transform=model_base.transform
    )
    return final_model


In [6]:
# 학습 루프 함수
def collate_fn(batch):
    # 배치 데이터 처리를 위한 함수
    return tuple(zip(*batch))

def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq =10):
    # 한 에폴 학습 함수
    model.train()# 모델을 훈련모드로
    prog_bar = tqdm(data_loader, desc =f'Epoch {epoch} [Train]')
    total_loss_epoch = 0
    num_batches = 0

    for images, targets in prog_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets) # 모델 forward 호출
        losses = sum(loss for loss in loss_dict.values()) # 모든 손실 합산 

        if not torch.isfinite(losses):
            print(f"Epoch {epoch}, Batch 건너뜀 : Loss is {losses.item()}")
            continue

        optimizer.zero_grad()
        losses.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        
        optimizer.step()

        total_loss_epoch += losses.item()
        num_batches +=1

        prog_bar.set_postfix(loss=losses.item(), lr = optimizer.param_groups[0]["lr"])
    avg_loss_epoch = total_loss_epoch / num_batches if num_batches > 0 else float('nan')
    return avg_loss_epoch

@torch.no_grad() # 평가 시에는 그래디언트 계산 불필요
def evaluate(model, data_loader, device):
    # 모델 평가 함수
    model.eval()
    total_loss_eval = 0
    num_batches = 0

    prog_bar = tqdm(data_loader, desc = '[Val]')
    for images, targets in prog_bar:
        images = list(image.to(device) for image in images)
        targets =[{k: v.to(device) for k, v in t.items()} for t in targets]

        #평가 시에도 model 호출하면, 학습과 동일한 손실 계산 (역전 파는 안함 )
        loss_dict,_ = model(images, targets)
        #losses = sum(loss for loss in loss_dict.values())
        if not loss_dict:
            continue

        losses = sum(loss for loss in loss_dict.values())
        
        if torch.isfinite(losses):
            total_loss_eval += losses.item()
            num_batches += 1
            prog_bar.set_postfix(loss=losses.item())
        else:
            prog_bar.set_postfix(loss=float('nan'))
            
    
    avg_loss_eval = total_loss_eval / num_batches if num_batches > 0 else float('nan')
    # 필요시  mAP 등 다른 평가 지표 계신 로직 추가 가능 
    return avg_loss_eval
    

In [7]:
#추론 및 시각화 클래스
class SurgicalToolDetector:
    # TorxhVisision  기반 수술도구 검출기 (수술 단계 예측 포함 )
    def __init__(self, model_path, parser_obj, device, confidence_threshold=0.3):
        self.device = device
        self.parser = parser_obj
        self.confidence_threshold = confidence_threshold

        num_tool_classes = len(self.parser.tool_categories)
        num_phases = self. parser.num_phases

        self.model = get_custom_surgical_mask_rcnn_model(num_tool_classes, num_phases)
        state_dict = torch.load(model_path, map_location = device)
        # 모델 가중치 로드 
        #checkpoint = torch.load(model_path, map_location = device)
        if 'model_state_dict' in state_dict:
            self.model.load_state_dict(state_dict['model_state_dict'])
        else:
            self.model.load_state_dict(state_dict)    
            
        self.model.to(device)
        self.model.eval() # 평가 모드
        print(f"모델 로드 완료 : {model_path}")
        """
        print(f"신뢰도 임계값 : {confidence_threshold}")
        print(f"도구 클래스 : {self.parser.idx_to_class}")
        if hasattr(self.parser, 'idx_to_phase'):
            print(f"단계 클래스 : {self.parser.idx_to_phase}")
        """
    def detect(self, image_path) : 
        # 단일 이미지 에서 수술 도구 검출 빛 단계 예특

        image_pil = Image.open(image_path).convert('RGB')
        transform_infer = T.Compose([
            T.ToTensor(),
        ])
        image_tensor = transform_infer(image_pil).to(self.device)
        #transform_func = get_tarnsform(train=False)

        
        #image_tensor,_=transform_func(image_pil, None)
        #image_tensor = transform_func(image_pil).to(self.device)
        
        with torch.no_grad():
            predictions  = self.model([image_tensor])

        return image_pil, predictions[0]

    def filter_predictions(self, pred_dict):
        # 신뢰도 임계값으로 예측 결과 필터링 
        scores = pred_dict['scores'].cpu().numpy()
        high_score_idx = scores >= self.confidence_threshold # 임계갑 이상

        masks = pred_dict['masks'][high_score_idx].cpu().numpy()
        if masks.ndim == 4 and masks.shape[1] == 1:
            masks = masks.squeeze(1)

        return {
            'boxes' : pred_dict['boxes'][high_score_idx].cpu().numpy(),
            'labels' : pred_dict['labels'][high_score_idx].cpu().numpy(),
            'scores' : scores[high_score_idx],
            'masks' : masks,
            'predicted_phase_idx' : pred_dict['predicted_phase_idx'].item(), #.item으로 스탈라값 추출
            'phase_probs' : pred_dict['phase_probs'].cpu().numpy(),               
        }
        #return filter_predictions

    def visualize_predictions(self, image_pil, filtered_pred_dict, save_path=None):
        # 검출 결과 시각화
        fig, ax = plt.subplots(1,1,figsize=(16,10)) # 크기 조정
        
        ax.imshow(image_pil)

        num_tool_categories = len(self.parser.tool_categories)
        #colors = plt.cm.get_cmap('tab20', num_tool_categories)
        colors = plt.get_cmap('tab20', num_tool_categories)

        for i in range(len(filtered_pred_dict['boxes'])):
            box = filtered_pred_dict['boxes'][i]
            label_idx_pred = filtered_pred_dict['labels'][i]
            score = filtered_pred_dict['scores'][i]
            mask_np = filtered_pred_dict['masks'][i] > 0.5

            class_idx_map = label_idx_pred - 1
            class_name = self.parser.idx_to_class.get(class_idx_map, f"Unknown_ID:{label_idx_pred}")
            color_val = colors(class_idx_map % num_tool_categories)

            rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=2, edgecolor = color_val, facecolor = 'none') 

            ax.add_patch(rect)
            
            # 마스크 오버레이
            color_mask = np.array(color_val)
            color_mask[3] = 0.4
            masked_image_area = np.zeros((*mask_np.shape, 4), dtype =np.float32)
            masked_image_area[mask_np] = color_mask
            ax.imshow(masked_image_area) # 이미지 위에 덧그리기
            
            ax.text(
                box[0], 
                box[1] -10,
                f'{class_name} : {score:.2f}',
                fontsize=9, color = 'white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor = color_val, alpha =0.9))

        #예측 된 수술 단께 정보 표시
        phase_idx = filtered_pred_dict['predicted_phase_idx']
        phase_name = self.parser.idx_to_phase.get(phase_idx, "N/A")
        phase_prob = filtered_pred_dict['phase_probs'][phase_idx]
        ax.set_title(f'Predicted Phase : {phase_name} (Prob: {phase_prob:.2f})', fontsize=12)
        ax.axis('off')
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches = 'tight')
            print(f"결과 저장 : {save_path}")
        plt.show(block=False)

    def get_detection_summary(self, filtered_pred_dict) : 
        #검출 결과 요약(도구 + 단계)
        summary = {'total_detected_tools': len(filtered_pred_dict['boxes']),
                   'detected_tools_info' : []
                  }
        for i in range(len(filtered_pred_dict['boxes'])):
            label_idx = filtered_pred_dict['labels'][i] - 1
            #score = filtered_pred_dict['scores'][i]
            class_name = self.parser.idx_to_class.get(label_idx, f"Unknown_ID:{label_idx + 1}")
            summary['detected_tools_info'].append({
                'tool_name':class_name,
                'confidence' : float(filtered_pred_dict['scores'][i]),
                'bbox_xyxy': filtered_pred_dict['boxes'][i].tolist()
            })
        phase_idx = filtered_pred_dict['predicted_phase_idx']
        summary['predicted_phase_info'] = {
                'phase_name': self.parser.idx_to_phase.get(phase_idx,"N/A"),
                'phase_confidence' : float(filtered_pred_dict['phase_probs'][phase_idx]),
                'predicted_phase_id': phase_idx
        }

        return summary
            
print("has filter_predictions : ", hasattr(SurgicalToolDetector,"visualize_predictions"))
print("has filter_predictions_fn : ", 'filter_predictions_fn' in globals())
#print('size=', fig.get_size_inches().tolist())

has filter_predictions :  True
has filter_predictions_fn :  False


In [8]:
def filter_predictions_fn(pred_dict,confidence_threshold=0.3):
    # 신뢰도 임계값으로 예측 결과 필터링 
    scores = pred_dict['scores'].detach().cpu().numpy()
    keep = scores >= confidence_threshold
    #high_score_idx = scores >= self.confidence_threshold # 임계갑 이상

    masks = pred_dict['masks'][keep].detach().cpu().numpy()
    if masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks.squeeze(1)

    return {
        'boxes' : pred_dict['boxes'][keep].detach().cpu().numpy(),
        'labels' : pred_dict['labels'][keep].detach().cpu().numpy(),
        'scores' : scores[keep],
        'masks' : masks,
        'predicted_phase_idx' : int(pred_dict['predicted_phase_idx'].item()
                                    if hasattr(pred_dict['predicted_phase_idx'],'item')
                                    else pred_dict['predicted_phase_idx']), #.item으로 스탈라값 추출
        'phase_probs' : pred_dict['phase_probs'].detach().cpu().numpy(),               
    }  

In [10]:
from pathlib import Path
import os, json, math
import numpy as np
import pandas as pd
#
def get_gt_phase_from_item(item, parser):
    gt_idx = int(item['phase_id'])
    gt_name = parser.idx_to_phase.get(gt_idx,"N/A")
    return gt_idx, gt_name

def ensure_dir(p:Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

@torch.no_grad()
def datch_infer_and_report(detector, dataset_items, val_items_subset,parser, 
                           max_images=2, save_root="inference_results_viz_batch_01",
                          confidence_threshold=0.3, mask_threshold=0.5):
    save_root = ensure_dir(Path(save_root))

    if hasattr(val_items_subset,"indices") and len(val_items_subset.indices) > 0:
        eval_indices = val_items_subset.indices[:max_images]
    else:
        eval_indices = list(range(min(max_images, len(dataset_items))))

    total = 0
    correct =0
    per_image_lines=[]
    print("\n-- 배치 추론 결과 --")
    for k, idx in enumerate(eval_indices,1):
        item = dataset_items[idx]
        image_path = item['image_path']
        gt_idx, gt_name = get_gt_phase_from_item(item,parser)

        try:
            pil_img, pred_out = detector.detect(image_path)
            filtered = detector.filter_predictions(pred_out)

        except Exception as e:
            print(f"[경고] 처리 실패 : {image_path} ({e})")
            total += 1
            per_image_lines.append({"image_path" : image_path, "error":str(e)})
            continue
        else:
            if 'predicted_phase_idx' not in filtered or 'phase_probs' not in filtered:
                total += 1 
                per_image_lines.append({"image_path" : image_path, "error":"phase fields missing"})
                continue

            pred_idx = int(filtered['predicted_phase_idx'])
            prob = float(filtered['phase_probs'][pred_idx])
            pred_name = parser.idx_to_phase.get(pred_idx,"N/A")
            
            save_path = Path(save_root)/ f"detected_{Path(image_path).name}"
            detector.visualize_predictions(pil_img,filtered, save_path = str(save_path))
            
            is_ok = (pred_idx == gt_idx)
            total += 1
            correct += int(is_ok)

            line = (
                f"[{k:02d}] 이미지: {image_path}\n"
                f"  실제 수술 단계: {gt_name}\n"    
                f"  예측 수술 단계: {pred_name} (신뢰도 : {prob:.2f})\n"
                f"  결과 이미지   : {save_path}\n")
            
            print(line)
            per_image_lines.append({
                "image_path" : image_path,
                "gt_phase" : gt_name,
                "pred_phase" : pred_name,
                "pred_prob" : prob,
                "correct" : bool(is_ok),
                "viz_path" : str(save_path),
            })

    acc = correct / total if total >0 else float('nan')
    print(f"\n 요약 : 정확도(Top-1) = {acc:.4f} (정답 {correct} / 총 {total})")

    out_json = Path(save_root) / "batchh_infer_summary_10_resnet50_fpn_01.json"
    with open(out_json, "w", encoding ='utf-8') as f:
        json.dump({
            "accuracy_top1" : acc,
            "num_correct" : int(correct),
            "num_total" : int(total),
            "pre_image" : per_image_lines,
        },f, ensure_ascii=False, indent =2)
    print(f"요약 저장:{out_json}")

    df = pd.DataFrame(per_image_lines)
    excel_path = Path(save_root) / "batchh_infer_summary_10_resnet50_fpn_01.xlsx"
    df.to_excel(excel_path,index=False)
    print(f"엑셀 완료 저장:{excel_path}")
    
    return acc, per_image_lines

###__________________####

In [13]:
# =================================================================
# === 🚀 통합 평가 모듈 (v2.0) 🚀 ===
# =================================================================
# 기능:
# 1. Top-1 단계 예측 정확도 계산
# 2. 신뢰도 보정(Calibration) 평가 (ECE, Brier) 및 그래프 저장
# 3. 도구 탐지 성능(mAP) 평가 (상세 IoU별 AP 포함)
# 4. 모든 결과를 하나의 JSON 파일과 개별 엑셀 파일로 저장

import json
import pandas as pd
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.ops import box_iou
from tqdm import tqdm

# --- 헬퍼(Helper) 함수들 ---

def ensure_dir(p: Path):
    """경로가 없으면 생성하는 함수"""
    p.mkdir(parents=True, exist_ok=True)
    return p

# --- 1. 수술 단계 예측 정확도 (Top-1 Accuracy) ---
@torch.no_grad()
def get_phase_accuracy(detector, dataset_items, val_items_subset, parser, max_images=30, save_root="evaluation_results_1_1/viz"):
    """배치 추론을 통해 수술 단계 예측의 Top-1 정확도를 계산하고, 개별 예측 결과를 반환합니다."""
    save_path = ensure_dir(Path(save_root))
    
    # 평가 대상 인덱스 결정
    if hasattr(val_items_subset, "indices") and len(val_items_subset.indices) > 0:
        eval_indices = val_items_subset.indices[:max_images]
    else:
        eval_indices = list(range(min(max_images, len(dataset_items))))

    total, correct = 0, 0
    per_image_results = []

    print(f"총 {len(eval_indices)}개의 검증 이미지에 대해 단계 예측 정확도를 계산합니다...")
    for idx in tqdm(eval_indices, desc="[단계 예측 평가]"):
        item = dataset_items[idx]
        image_path = item['image_path']
        gt_idx = int(item['phase_id'])
        gt_name = parser.idx_to_phase.get(gt_idx, "N/A")
        
        try:
            pil_img, pred_out = detector.detect(image_path)
            filtered = detector.filter_predictions(pred_out)

            if 'predicted_phase_idx' not in filtered:
                raise ValueError("모델 출력에서 'predicted_phase_idx'를 찾을 수 없습니다.")

            pred_idx = int(filtered['predicted_phase_idx'])
            prob = float(filtered['phase_probs'][pred_idx])
            pred_name = parser.idx_to_phase.get(pred_idx, "N/A")

            is_correct = (pred_idx == gt_idx)
            total += 1
            correct += int(is_correct)
            
            # 시각화 결과 저장
            viz_save_path = save_path / f"detected_{Path(image_path).name}"
            detector.visualize_predictions(pil_img, filtered, save_path=str(viz_save_path))
            plt.close() # 메모리 관리를 위해 그래프 창 닫기

            per_image_results.append({
                "image_path": image_path, "gt_phase": gt_name,
                "pred_phase": pred_name, "pred_prob": prob,
                "correct": bool(is_correct), "viz_path": str(viz_save_path),
            })

        except Exception as e:
            print(f"[경고] {image_path} 처리 중 오류: {e}")
            per_image_results.append({"image_path": image_path, "error": str(e)})

    accuracy = correct / total if total > 0 else 0.0
    return accuracy, per_image_results

# --- 2. 신뢰도 보정 평가 (Calibration: ECE, Brier) ---
@torch.no_grad()
def collect_phase_outputs(model, data_loader, device):
    model.eval()
    all_logits, all_probs, all_labels = [], [], []
    for images, targets in tqdm(data_loader, desc="[Calibration 데이터 수집]"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        _, detections = model(images, targets)
        for det, tgt in zip(detections, targets):
            all_logits.append(det['phase_logits'].detach().cpu().float().unsqueeze(0))
            all_probs.append(det['phase_probs'].detach().cpu().float().unsqueeze(0))
            all_labels.append(tgt['phase_label'].detach().cpu().long().view(1))
            
    if not all_logits: return torch.empty(0), torch.empty(0), torch.empty(0)
    return torch.cat(all_logits), torch.cat(all_probs), torch.cat(all_labels)

def compute_ece_and_bins(probs, labels, n_bins=15):
    if probs.numel() == 0: return 0.0, []
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    ece, bin_stats = torch.tensor(0.0), []
    N = probs.shape[0]
    for i in range(n_bins):
        lo, hi = bins[i].item(), bins[i+1].item()
        mask = (conf >= lo) & (conf < hi) if i < n_bins - 1 else (conf >= lo) & (conf <= hi)
        n_b = mask.sum().item()
        if n_b == 0:
            bin_stats.append(((lo + hi) / 2, 0.0, 0.0, 0))
            continue
        acc_b = correct[mask].mean().item()
        conf_b = conf[mask].mean().item()
        ece += (n_b / N) * abs(acc_b - conf_b)
        bin_stats.append(((lo + hi) / 2, acc_b, conf_b, n_b))
    return float(ece.item()), bin_stats

def plot_reliability(bin_stats, title="Reliability Diagram", save_path=None):
    if not bin_stats: return
    centers, accs, confs = [b[0] for b in bin_stats], [b[1] for b in bin_stats], [b[2] for b in bin_stats]
    width = (centers[1] - centers[0]) * 0.8 if len(centers) > 1 else 0.06
    
    plt.figure(figsize=(6, 6), dpi=100)
    plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
    plt.bar(centers, accs, width=width, alpha=0.6, label='Observed accuracy')
    plt.plot(centers, confs, 'o-', color='C1', label='Mean confidence')
    plt.xlabel('Confidence', fontsize=12); plt.ylabel('Accuracy', fontsize=12)
    plt.title(title, fontsize=14, weight='bold')
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)
    plt.xlim(0, 1); plt.ylim(0, 1)
    
    if save_path:
        save_path_obj = Path(save_path)
        save_path_obj.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path_obj, bbox_inches='tight')
        print(f"📊 보정 그래프 저장 완료: {save_path_obj}")
    plt.show()
    plt.close()

def brier_score(probs, labels, num_classes):
    if probs.numel() == 0: return float('nan')
    y = torch.zeros_like(probs); y[torch.arange(probs.size(0)), labels] = 1.0
    return float(((probs - y) ** 2).mean().item())

def fit_temperature(logits, labels, max_iter=200):
    scaler = TemperatureScaler().to(logits.device)
    optimizer = torch.optim.LBFGS(scaler.parameters(), lr=0.5, max_iter=max_iter)
    criterion = torch.nn.CrossEntropyLoss()
    def closure():
        optimizer.zero_grad()
        loss = criterion(scaler(logits), labels)
        loss.backward()
        return loss
    optimizer.step(closure)
    return scaler

class TemperatureScaler(torch.nn.Module):
    def __init__(self, init_T=1.0):
        super().__init__()
        self.log_T = torch.nn.Parameter(torch.log(torch.tensor([init_T], dtype=torch.float32)))
    @property
    def T(self): return torch.exp(self.log_T)
    def forward(self, logits): return logits / self.T

@torch.no_grad()
def get_calibration_scores(model, val_loader, device, save_dir):
    logits, probs, labels = collect_phase_outputs(model, val_loader, device)
    if logits.numel() == 0: return None
    
    ece_before, bins_before = compute_ece_and_bins(probs, labels)
    brier_before = brier_score(probs, labels, probs.size(1))
    plot_reliability(bins_before, title="Phase Reliability (Before TS)", save_path=Path(save_dir) / "reliability_before_ts.png")

    scaler = fit_temperature(logits.to(device), labels.to(device))
    probs_after = torch.softmax(scaler(logits.to(device)), dim=1).cpu()
    ece_after, bins_after = compute_ece_and_bins(probs_after, labels)
    brier_after = brier_score(probs_after, labels, probs_after.size(1))
    plot_reliability(bins_after, title=f"Phase Reliability (After TS, T={scaler.T.item():.2f})", save_path=Path(save_dir) / "reliability_after_ts.png")

    return {
        "before_scaling": {"ECE": ece_before, "Brier_Score": brier_before},
        "after_scaling": {"ECE": ece_after, "Brier_Score": brier_after, "Temperature": scaler.T.item()}
    }

# --- 3. 도구 탐지 성능 (mAP) ---
@torch.no_grad()
def evaluate_map(model, data_loader, device, iou_thresholds=None, max_batches=None):
    model.eval()
    if iou_thresholds is None:
        iou_thresholds = torch.arange(0.5, 1.0, 0.05,device=device)
        #iou_thresholds = np.arange(0.5, 1.0, 0.05)
    
    all_preds = []
    all_targets = []
    
    # 데이터 수집
    batches_to_run = max_batches if max_batches else len(data_loader)
    for i, (images, targets) in enumerate(tqdm(data_loader, total=batches_to_run, desc="[mAP 데이터 수집]")):
        if max_batches and i >= max_batches: break
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        outputs = model(images)
        
        all_preds.extend(outputs)
        all_targets.extend(targets)
        
    # 클래스별 AP 계산
    aps_per_iou = defaultdict(list)
    for iou_thr in tqdm(iou_thresholds, desc="[mAP 계산]"):
        aps_per_class = []
        for c in range(1, model.num_tool_classes + 1): # 클래스별로 순회 (배경 제외)
            preds_c_indices = [i for i, p in enumerate(all_preds) if (p['labels']==c).any()]

            if not preds_c_indices:
                aps_per_class.append(0.0)
                continue
                
            num_gt = sum([(t['labels'] == c).sum() for t in all_targets])

            pred_scores_c = torch.cat([all_preds[i]['scores'][all_preds[i]['labels']==c]for i in preds_c_indices])
            pred_boxes_c = torch.cat([all_preds[i]['boxes'][all_preds[i]['labels']==c]for i in preds_c_indices])

            pred_img_ids = torch.tensor([i for i in preds_c_indices for _ in range((all_preds[i]['labels']==c).sum())], device=device)

            # 신뢰도 순으로 정렬
            sort_idx = torch.argsort(pred_scores_c, descending=True)
            pred_boxes_c = pred_boxes_c[sort_idx]
            pred_img_ids = pred_img_ids[sort_idx]
            
            tp = torch.zeros(len(pred_boxes_c), device=device,dtype=torch.bool)
            fp = torch.zeros(len(pred_boxes_c), device=device,dtype=torch.bool)
            
            gt_detected = [torch.zeros((t['labels']==c).sum(),dtype=torch.bool,device=device) for t in all_targets]
            

            for i in range(len(pred_boxes_c)):
                img_id = pred_img_ids[i].item()
                gt_boxes_img = all_targets[img_id]['boxes'][all_targets[img_id]['labels']==c]
                
                if len(gt_boxes_img) == 0:
                    fp[i] = 1; continue
                    
                ious = box_iou(pred_boxes_c[i].unsqueeze(0), gt_boxes_img)
                max_iou, max_idx = ious.max(1)
                
                if max_iou >= iou_thr:
                    if not gt_detected[img_id][max_idx]:
                        tp[i] = 1
                        gt_detected[img_id][max_idx] = True
                    else:
                        fp[i] = 1
                else:
                    fp[i] = 1
            
            tp_cumsum = torch.cumsum(tp, dim=0)
            fp_cumsum = torch.cumsum(fp, dim=0)
            
            recalls = tp_cumsum / (num_gt + 1e-6)
            precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
            
            precisions = torch.cat([torch.tensor([1.0],device=device),recalls])
            recalls = torch.cat([torch.tensor([0.0], device=device),recalls])
            
            ap = torch.trapz(precisions, recalls).item()
            aps_per_class.append(ap)
        
        ap_for_iou = np.mean(aps_per_class) if aps_per_class else 0.0
        aps_per_iou[f"{iou_thr.item():.2f}"] = ap_for_iou
        
    mAP_final = np.mean(list(aps_per_iou.values()))
    return mAP_final, aps_per_iou

# --- 메인 통합 평가 함수 ---
def perform_full_evaluation(model, detector, dataset_items, val_items_subset, val_loader, parser, device, save_dir="evaluation_results_1_1"):
    print("=" * 25, "통합 평가 시작", "=" * 25)
    save_path = ensure_dir(Path(save_dir))
    
    # 1. 수술 단계 예측 정확도 (Top-1 Accuracy)
    print("\n[1/3] 수술 단계 예측 정확도(Top-1) 및 개별 결과 생성 중...")
    phase_accuracy, phase_predictions = get_phase_accuracy(detector, dataset_items, val_items_subset, parser, max_images=30, save_root=str(save_path / "viz"))
    print(f"-> 단계 예측 정확도: {phase_accuracy:.4f}")

    # 2. 신뢰도 보정 평가 (ECE, Brier Score)
    print("\n[2/3] 신뢰도 보정(Calibration) 평가 중...")
    calibration_scores = get_calibration_scores(model, val_loader, device, save_dir=str(save_path / "plots"))
    if calibration_scores:
        print(f"-> ECE (보정 전): {calibration_scores['before_scaling']['ECE']:.4f} -> (보정 후): {calibration_scores['after_scaling']['ECE']:.4f}")

    # 3. 도구 탐지 성능 평가 (mAP)
    print("\n[3/3] 도구 탐지 성능(mAP) 평가 중...")
    mAP, per_iou_aps = evaluate_map(model, val_loader, device)
    print(f"-> 최종 mAP@[.5:.95]: {mAP:.4f}")
    
    # --- 결과 종합 및 저장 ---
    final_results = {
        "summary": {
            "phase_top1_accuracy": phase_accuracy,
            "detection_mAP_0.5_0.95": mAP,
            "detection_mAP_per_iou": per_iou_aps, # 상세 AP 추가
            "phase_calibration_scores": calibration_scores,
        },
        "individual_phase_predictions": phase_predictions,
    }
    
    # JSON 파일 저장
    json_path = save_path / "full_evaluation_results_01.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        def default_converter(o):
            if isinstance(o, (np.integer, np.floating)): return o.item()
            if isinstance(o, (torch.Tensor)): return o.tolist()
            if isinstance(o, Path): return str(o)
            raise TypeError(f"Object of type {o.__class__.__name__} is not JSON serializable")
        json.dump(final_results, f, ensure_ascii=False, indent=2, default=default_converter)
    print(f"\n- ✅ 전체 평가 결과가 JSON 파일로 저장되었습니다: {json_path}")
    
    # 엑셀 파일 저장
    if phase_predictions:
        excel_path = save_path / "individual_phase_predictions_01.xlsx"
        df = pd.DataFrame(phase_predictions)
        df.to_excel(excel_path, index=False)
        print(f"- ✅ 개별 예측 결과가 엑셀 파일로 저장되었습니다: {excel_path}")
    
    print("\n" + "=" * 26, "통합 평가 완료", "=" * 26)
    return final_results


In [1]:
# =================================================================
# === 🚀 통합 평가 실행 ===
# =================================================================
# 학습 완료 후, 최고 성능 모델(model_best.pth)을 기준으로 평가를 수행합니다.

print("\n === 통합 평가 테스트 시작 ===")

# best_model_path가 정의되어 있는지 확인
if 'best_model_path' not in locals() or not best_model_path.exists():
    print(f"경고: 최고 성능 모델 파일('model_best.pth')을 찾을 수 없어 평가를 건너뜁니다.")
    print("학습을 먼저 완료해주세요.")
elif not val_dataset or len(val_dataset) == 0:
    print("경고: 검증 데이터 셋이 없어 평가를 건너뜁니다.")
else:
    # 1. 추론기(detector) 인스턴스 생성
    #    (메모리 관리를 위해 기존에 있다면 재생성)
    if 'detector' in locals():
        del detector
        torch.cuda.empty_cache()
        
    print(f"평가용 모델 로딩: {best_model_path}")
    detector = SurgicalToolDetector(
        model_path=str(best_model_path),
        parser_obj=parser,
        device=device,
        confidence_threshold=0.3  # 평가 시 사용할 신뢰도 임계값
    )
    
    # 2. 통합 평가 함수 호출
    #    - model 객체는 학습 마지막 상태의 모델을 그대로 사용
    #    - detector는 best 모델 가중치를 로드한 추론기
    evaluation_results = perform_full_evaluation(
        model=model,                  # 현재 메모리에 있는 모델
        detector=detector,            # 최고 성능 가중치를 로드한 추론기
        dataset_items=dataset_items,      # 전체 데이터 아이템
        val_items_subset=val_items_subset, # 검증용 Subset
        val_loader=val_loader,        # 검증용 데이터로더
        parser=parser,                # 데이터 파서
        device=device,
        save_dir="evaluation_results_1_1" # 결과 저장 폴더
    )



 === 통합 평가 테스트 시작 ===
경고: 최고 성능 모델 파일('model_best.pth')을 찾을 수 없어 평가를 건너뜁니다.
학습을 먼저 완료해주세요.


In [ ]:
-> 단계 예측 정확도: 0.3333

[2/3] 신뢰도 보정(Calibration) 평가 중...
[Calibration 데이터 수집]: 100%|██████████████| 977/977 [02:37<00:00,  6.19it/s]
📊 보정 그래프 저장 완료: evaluation_results_1_1/plots/reliability_before_ts.png

📊 보정 그래프 저장 완료: evaluation_results_1_1/plots/reliability_after_ts.png

-> ECE (보정 전): 0.2604 -> (보정 후): 0.0438

[3/3] 도구 탐지 성능(mAP) 평가 중...

In [ ]:
"""
# --- 3. 도구 탐지 성능 (mAP) ---
@torch.no_grad()
def evaluate_map(model, data_loader, device, iou_thresholds=None, max_batches=None):
    model.eval()
    if iou_thresholds is None:
        iou_thresholds = np.arange(0.5, 1.0, 0.05)
    
    all_preds = []
    all_targets = []
    
    # 데이터 수집
    batches_to_run = max_batches if max_batches else len(data_loader)
    for i, (images, targets) in enumerate(tqdm(data_loader, total=batches_to_run, desc="[mAP 데이터 수집]")):
        if max_batches and i >= max_batches: break
        images = [img.to(device) for img in images]
        outputs = model(images)
        all_preds.extend([ {k: v.cpu() for k,v in p.items()} for p in outputs ])
        all_targets.extend([ {k: v.cpu() for k,v in t.items()} for t in targets ])
        
    # 클래스별 AP 계산
    aps_per_iou = defaultdict(list)
    for iou_thr in tqdm(iou_thresholds, desc="[mAP 계산]"):
        aps_per_class = []
        for c in range(1, model.num_tool_classes + 1): # 클래스별로 순회 (배경 제외)
            preds_c = [p for p in all_preds if c in p['labels']]
            targets_c = [t for t in all_targets if c in t['labels']]
            
            if not preds_c: continue
            
            gt_boxes_c = [t['boxes'][t['labels']==c] for t in targets_c]
            
            pred_scores_c, pred_boxes_c, pred_img_ids = [], [], []
            for i, p in enumerate(all_preds):
                mask = (p['labels'] == c)
                if torch.any(mask):
                    pred_scores_c.extend(p['scores'][mask])
                    pred_boxes_c.extend(p['boxes'][mask])
                    pred_img_ids.extend([i] * mask.sum())

            if not pred_scores_c:
                aps_per_class.append(0.0)
                continue

            # 신뢰도 순으로 정렬
            sort_idx = torch.argsort(torch.tensor(pred_scores_c), descending=True)
            pred_boxes_c = torch.stack(pred_boxes_c)[sort_idx]
            pred_img_ids = torch.tensor(pred_img_ids)[sort_idx]
            
            tp = torch.zeros(len(pred_scores_c))
            fp = torch.zeros(len(pred_scores_c))
            gt_detected = [torch.zeros(len(b)) for b in gt_boxes_c]
            
            num_gt = sum(len(b) for b in gt_boxes_c)

            for i in range(len(pred_boxes_c)):
                img_id = pred_img_ids[i]
                gt_boxes_img = all_targets[img_id]['boxes'][all_targets[img_id]['labels']==c]
                
                if len(gt_boxes_img) == 0:
                    fp[i] = 1; continue
                    
                ious = box_iou(pred_boxes_c[i].unsqueeze(0), gt_boxes_img)
                max_iou, max_idx = ious.max(1)
                
                if max_iou >= iou_thr:
                    if gt_detected[img_id][max_idx] == 0:
                        tp[i] = 1
                        gt_detected[img_id][max_idx] = 1
                    else:
                        fp[i] = 1
                else:
                    fp[i] = 1
            
            tp_cumsum = torch.cumsum(tp, dim=0)
            fp_cumsum = torch.cumsum(fp, dim=0)
            
            recalls = tp_cumsum / (num_gt + 1e-6)
            precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
            
            precisions = torch.cat([torch.tensor([1.0]), precisions])
            recalls = torch.cat([torch.tensor([0.0]), recalls])
            
            ap = torch.trapz(precisions, recalls)
            aps_per_class.append(ap.item())
        
        ap_for_iou = np.mean(aps_per_class) if aps_per_class else 0.0
        aps_per_iou[f"{iou_thr:.2f}"] = ap_for_iou
        
    mAP_final = np.mean(list(aps_per_iou.values()))
    return mAP_final, aps_per_iou

# --- 메인 통합 평가 함수 ---
def perform_full_evaluation(model, detector, dataset_items, val_items_subset, val_loader, parser, device, save_dir="evaluation_results"):
    print("=" * 25, "통합 평가 시작", "=" * 25)
    save_path = ensure_dir(Path(save_dir))
    
    # 1. 수술 단계 예측 정확도 (Top-1 Accuracy)
    print("\n[1/3] 수술 단계 예측 정확도(Top-1) 및 개별 결과 생성 중...")
    phase_accuracy, phase_predictions = get_phase_accuracy(detector, dataset_items, val_items_subset, parser, max_images=30, save_root=str(save_path / "viz"))
    print(f"-> 단계 예측 정확도: {phase_accuracy:.4f}")

    # 2. 신뢰도 보정 평가 (ECE, Brier Score)
    print("\n[2/3] 신뢰도 보정(Calibration) 평가 중...")
    calibration_scores = get_calibration_scores(model, val_loader, device, save_dir=str(save_path / "plots"))
    if calibration_scores:
        print(f"-> ECE (보정 전): {calibration_scores['before_scaling']['ECE']:.4f} -> (보정 후): {calibration_scores['after_scaling']['ECE']:.4f}")

    # 3. 도구 탐지 성능 평가 (mAP)
    print("\n[3/3] 도구 탐지 성능(mAP) 평가 중...")
    mAP, per_iou_aps = evaluate_map(model, val_loader, device)
    print(f"-> 최종 mAP@[.5:.95]: {mAP:.4f}")
    
    # --- 결과 종합 및 저장 ---
    final_results = {
        "summary": {
            "phase_top1_accuracy": phase_accuracy,
            "detection_mAP_0.5_0.95": mAP,
            "detection_mAP_per_iou": per_iou_aps, # 상세 AP 추가
            "phase_calibration_scores": calibration_scores,
        },
        "individual_phase_predictions": phase_predictions,
    }
    
    # JSON 파일 저장
    json_path = save_path / "full_evaluation_results_01.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        def default_converter(o):
            if isinstance(o, (np.integer, np.floating)): return o.item()
            if isinstance(o, (torch.Tensor)): return o.tolist()
            if isinstance(o, Path): return str(o)
            raise TypeError(f"Object of type {o.__class__.__name__} is not JSON serializable")
        json.dump(final_results, f, ensure_ascii=False, indent=2, default=default_converter)
    print(f"\n- ✅ 전체 평가 결과가 JSON 파일로 저장되었습니다: {json_path}")
    
    # 엑셀 파일 저장
    if phase_predictions:
        excel_path = save_path / "individual_phase_predictions_01.xlsx"
        df = pd.DataFrame(phase_predictions)
        df.to_excel(excel_path, index=False)
        print(f"- ✅ 개별 예측 결과가 엑셀 파일로 저장되었습니다: {excel_path}")
    
    print("\n" + "=" * 26, "통합 평가 완료", "=" * 26)
    return final_results
"""